# Self-Reflection & Critique — with AutoGen agents

The same idea as the other notebook, but the two jobs are done by **agents** instead of function calls:

- a **critic agent** that scores the draft,
- a **writer agent** that rewrites it,
- and a **controller** — our own plain Python — that decides when to stop.

Everything is in this notebook.

**You need:** an OpenAI API key. A full run costs a cent or two.

**How to use it:** Runtime → Run all. The last cell asks you to paste a draft.

## What an agent actually gives you

Be honest about this, because it is the whole point.

| | plain function | AutoGen agent |
|---|---|---|
| the prompt | same | same |
| the model | same | same |
| the cost | same | same |
| the answer | same | same |
| **has a name others can talk to** | no | **yes** |
| **remembers earlier turns by itself** | no | **yes** |
| **can be dropped into a team** | no | **yes** |

The middle row is the one you feel straight away. In the other notebook you had to collect each lesson and paste it into the next prompt yourself. An agent keeps its own memory, so **the critic just remembers what it already told the writer**. Step 9 proves it.

## Step 1 — Install and import

- **`autogen_agentchat`** — gives us the `AssistantAgent` class.
- **`autogen_ext`** — connects an agent to OpenAI.
- **`json`** — turns the critic's reply into a Python dictionary. Comes with Python.

The install takes about half a minute the first time.

In [ ]:
%pip install -q "autogen-agentchat>=0.4" "autogen-ext[openai]>=0.4"

import json
import textwrap

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

print("Ready.")

## Step 2 — Your API key

Either add a Colab secret called `OPENAI_API_KEY` (🔑 icon on the left, switch on *Notebook access*), or run this cell and paste it. **`getpass`** hides it as you type.

In [ ]:
import getpass
import os

key = os.environ.get("OPENAI_API_KEY")

if not key:
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        key = None

if not key:
    key = getpass.getpass("Paste your OpenAI API key (hidden): ").strip()

MODEL = "gpt-4o-mini"
PASS_MARK = 8
MAX_ROUNDS = 3

print("Key loaded. Model:", MODEL)

## Step 3 — The rules we judge against

Same three pieces of text as the other notebook. They never change during a run.

In [ ]:
BRIEF = """
Write a short launch email for working professionals thinking about a live
weekend course in Generative AI and Agentic AI.

Who reads it:
- Software engineers, data analysts, DevOps people, tech managers
- They are busy, they dislike hype, and they want proof it is practical

Rules:
- 140 to 180 words
- Warm but professional
- Mention live classes, hands-on projects, and why it helps their career
- Do not promise salaries, jobs or placement
- Finish with one clear next step
"""

CHECKLIST = """
Give each of these a score out of 10:

1. Audience fit   - speaks to working professionals, no hype
2. Specificity    - real details, not vague claims
3. Rules followed - word count, tone, and everything the brief asked for
4. Evidence       - no promises it cannot back up
5. Next step      - ends with one clear thing to do
"""

PRINCIPLES = """
- Never invent facts, rankings, salaries or placements.
- Prefer concrete detail over general enthusiasm.
- Remember the reader is busy and hard to impress.
- Say what to change. Never give a vague opinion.
"""

print("Rules loaded.")

## Step 4 — Build the two agents

Each agent needs its own connection, because they need different settings:

- the **critic** runs at `temperature=0` and is put into JSON mode, so its reply is always valid JSON,
- the **writer** runs warmer and replies with ordinary text.

Look at what each one is told. The critic gets the checklist. **The writer does not** — if it knew how it was being scored, it would write to please the score instead of the reader.

In [ ]:
critic_connection = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=key,
    temperature=0,
    response_format={"type": "json_object"},   # replies are always valid JSON
)

writer_connection = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=key,
    temperature=0.4,
)

REPLY_FORMAT = """
Reply with JSON using exactly these keys:
  "score"   - one number out of 10, the lowest of the five checklist scores
  "summary" - one sentence on the state of the draft
  "issues"  - a list of at most 5 problems. Each one has:
                "criterion" - which checklist line it is about
                "severity"  - "minor", "major" or "blocking"
                "problem"   - what is wrong, quoting the draft
                "fix"       - the exact change to make
  "lesson"  - one sentence worth remembering for the next round
"""

critic = AssistantAgent(
    name="critic",
    model_client=critic_connection,
    system_message=(
        "You are a strict but fair editor. Judge every draft you are sent "
        "against the brief, the checklist and the principles below. "
        "Never praise vague writing. Never invent facts. Quote the words at fault.\n\n"
        f"BRIEF:\n{BRIEF}\n\nCHECKLIST:\n{CHECKLIST}\n\nPRINCIPLES:\n{PRINCIPLES}\n"
        f"{REPLY_FORMAT}"
    ),
)

writer = AssistantAgent(
    name="writer",
    model_client=writer_connection,
    system_message=(
        "You are a rewriting editor. You will be sent a draft and a list of "
        "changes. Apply them. Reply with the rewritten draft only - no notes.\n\n"
        f"BRIEF:\n{BRIEF}\n\nPRINCIPLES TO KEEP:\n{PRINCIPLES}"
    ),
)

print("Agents ready:", critic.name, "and", writer.name)

## Step 5 — Talking to each agent

`agent.run(task=...)` sends a message and gives back everything the agent said. The last message is its reply.

These are `async` functions, so we write `await` when we call them. That is just how AutoGen works — Colab handles it for you.

In [ ]:
async def ask_critic(draft):
    """Send a draft to the critic agent. Returns a plain dictionary."""
    reply = await critic.run(task=f"Judge this draft:\n\n{draft}")
    return json.loads(reply.messages[-1].content)


async def ask_writer(draft, report):
    """Send a draft and the list of fixes to the writer agent."""
    changes = "\n".join("- " + issue["fix"] for issue in report["issues"])
    reply = await writer.run(
        task=f"CURRENT DRAFT:\n{draft}\n\nCHANGES TO MAKE:\n{changes}"
    )
    return str(reply.messages[-1].content).strip()


print("Ready to talk to both agents.")

## Step 6 — Who decides when to stop

**Our code decides, not the agents.** A draft passes only if it hits the pass mark **and** has no `blocking` problem. A score of 9 with a blocking problem still fails.

In [ ]:
def has_passed(report):
    """Decide if a draft is good enough. Our decision, not the agent's."""
    blocking = [i for i in report["issues"] if i["severity"] == "blocking"]
    return report["score"] >= PASS_MARK and len(blocking) == 0


def heading(left, right=""):
    """Print a title bar."""
    print("\n" + "=" * 70)
    print(left + " " * max(1, 70 - len(left) - len(right)) + right if right else left)
    print("=" * 70)


def paragraph(text):
    """Print text wrapped to 70 characters, using Python's textwrap."""
    for line in str(text).split("\n"):
        print(textwrap.fill(line, width=70) if line.strip() else "")


print("Helpers ready.")

## Step 7 — The controller

This is the loop that passes work between the two agents. **It is not an agent.** It is ordinary Python with a counter and a threshold, because a model that can be talked into one more round, will be.

It stops when any one of these is true:

1. the draft passed,
2. we used all our rounds,
3. the score stopped going up.

In [ ]:
async def improve(draft):
    """Pass work between the two agents until the draft passes or rounds run out."""
    heading("STEP 1 - YOUR DRAFT", f"{len(draft.split())} words")
    paragraph(draft)

    scores = []
    step = 1
    last_score = None
    passed = False

    for round_number in range(1, MAX_ROUNDS + 1):
        report = await ask_critic(draft)
        passed = has_passed(report)
        scores.append(report["score"])

        step += 1
        name = "CRITIC AGENT SCORES YOUR DRAFT" if last_score is None else "CRITIC AGENT SCORES THE REWRITE"
        verdict = "PASSED" if passed else f"below {PASS_MARK}"
        moved = "" if last_score is None else f"  ({report['score'] - last_score:+d})"
        heading(f"STEP {step} - {name}", f"{report['score']}/10  {verdict}{moved}")

        print(report["summary"])
        print(f"\nWhat it wants changed ({len(report['issues'])}):")
        for number, issue in enumerate(report["issues"], start=1):
            print(f"  {number}. [{issue['severity']}] {issue['criterion']}")
            print(textwrap.fill(issue["fix"], width=65,
                                initial_indent="     ", subsequent_indent="     "))
        print(f"\nLesson it will remember: {report['lesson']}")

        if passed:
            break
        if last_score is not None and report["score"] <= last_score:
            print("\n(The score stopped going up, so we stop here.)")
            break
        last_score = report["score"]

        step += 1
        draft = await ask_writer(draft, report)
        heading(f"STEP {step} - WRITER AGENT REWRITES IT", f"{len(report['issues'])} changes")
        paragraph(draft)

    heading("HOW THE SCORE MOVED")
    for number, score in enumerate(scores, start=1):
        print(f"  round {number}   {score:>2}/10   {'#' * score}")

    if passed:
        heading(f"RESULT: PASSED - {scores[-1]}/10", "scores " + " -> ".join(str(s) for s in scores))
        print("\nFINAL DRAFT\n")
    else:
        heading(f"RESULT: NOT PASSED - best {max(scores)}/10, needed {PASS_MARK}/10",
                "scores " + " -> ".join(str(s) for s in scores))
        print("\nThe draft below is the rewrite made after the last score.")
        print("It was never scored - we ran out of rounds first.\n")
        print("DRAFT AS IT STANDS - NOT PASSED\n")

    paragraph(draft)
    return draft


print("Controller ready.")

## Step 8 — Run it

Paste a draft and press Enter once. Watch the two agents pass work back and forth.

If you have nothing ready, use this one:

```
AI is changing the world and you should learn it as soon as possible. Our course will make you skilled in all AI tools and help you get better opportunities. Join now to become future ready.
```

In [ ]:
draft = input("Paste your draft here, then press Enter:\n\n").strip()

if not draft:
    raise ValueError("Nothing was entered. Run this cell again and paste a draft.")

print(f"\n{len(draft.split())} words. Sending it to the critic agent...")

final_draft = await improve(draft)

## Step 9 — What the critic remembered

Here is the thing you did not have to build.

Ask the critic a follow-up question. It already knows what it said earlier, because an agent keeps its own conversation. In the other notebook you would have had to collect every earlier critique and paste it into this prompt yourself.

In [ ]:
reply = await critic.run(
    task=(
        "Without re-reading the drafts, what did you ask to be changed across this "
        "whole session, and which of your instructions were actually applied? "
        "Reply as JSON with one key called 'points' holding a list of short sentences."
    )
)

answer = json.loads(reply.messages[-1].content)

for point in answer["points"]:
    paragraph("- " + str(point))

## Step 10 — Letting the framework run the conversation

Above, we passed work between the agents ourselves. AutoGen can do that part for you: give `RoundRobinGroupChat` a list of agents and a stopping rule, and it takes turns automatically.

Run it and compare. It is less code. But notice what the stopping rule became — a **message count**, not "score of 8 with nothing blocking". For a quality gate, that matters.

**Rule of thumb:** let the framework run the conversation. Keep your own code in charge of anything that costs money or reaches a customer.

In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [writer, critic],
    termination_condition=MaxMessageTermination(5),
)

result = await team.run(task=f"Improve this draft:\n\n{draft}")

heading("THE SAME TWO AGENTS, AS A TEAM", f"{len(result.messages)} messages")
for message in result.messages:
    print(f"\n--- {message.source} ---")
    paragraph(str(message.content)[:400])

print("\nIt stopped because it hit the message limit,")
print("not because the draft was good enough. That is the trade-off.")

In [ ]:
# Close the connections when you are finished.
await critic_connection.close()
await writer_connection.close()
print("Closed.")

## Try these next

1. **Change `PASS_MARK` to 9** in Step 2 and run the same draft. More rounds? A better answer, or just a longer one?
2. **Give the writer the checklist** by pasting `CHECKLIST` into its system message. Scores go up. Then read the draft.
3. **Delete the principles** from the critic and run a draft full of invented statistics. Watch which problems stop being reported.
4. **Add a third agent** — a fact checker that only reports claims with nothing behind them — and send the draft to it before the writer.

## What to take away

- The agents did not make the critique better. **The checklist did.** AutoGen changed where the job lives, not how well it is done.
- What you got for free was **memory**. Each agent remembers its own turns, so you stopped assembling that by hand.
- What you must not hand over is **the stopping rule**. The controller stayed as ordinary Python on purpose.
- Rewriting fixes how something is *said*. It cannot supply a fact the model never had.